# 38 · Quem usa a rede social do app é mais ativo?

Três sinais de uso social, nos três fatores (ativos, engajamento e churn):
- **posts no feed** (com data; curtidas não são usadas, decisoes.md): posts nos 90 dias antes da data de referência;
- **rede**: relações mútuas (seguir e ser seguido de volta). As relações só se formam nos primeiros
  ~4 meses da conta (decisoes.md), então a rede é comparada dentro do mesmo tempo de casa;
- **treino acompanhado** (`treino_com`): treinos feitos com outra conta nos 90 dias.

Cuidado de leitura: o post é sobre um treino (77% dos posts de 2026 caem em dia com treino da conta,
cerca de 6 posts a cada 100 treinos). Quem treina mais tem mais o que postar, então a comparação
que importa é com o mesmo volume de treino.

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import NIVEIS

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")

usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "sistema"])
assin = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "plano_atual", "inicio", "ativa_no_corte", "renovacao_automatica"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "treino_com"])
posts = pd.read_parquet(P / "feed_posts_limpa.parquet", columns=["user_id", "data"])
rede = pd.read_parquet(P / "rede_por_conta.parquet", columns=["user_id", "n_seguidores", "n_seguindo", "n_mutuas"])
rede_mes = pd.read_parquet(P / "rede_mes.parquet", columns=["user_id", "mes", "n_mutuas"])
eng = pd.read_parquet(P / "engajamento_corte.parquet")
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "ativa_inicio_mes", "perdida_no_mes"])
ativ["acompanhado"] = ativ.treino_com.notna()
print(len(posts), "posts de", posts.data.min().date(), "a", posts.data.max().date(), "|", len(ativ), "treinos |", int(ativ.acompanhado.sum()), "acompanhados")
p26 = posts[posts.data >= "2026-01-01"]
m = p26.merge(ativ[["user_id", "data"]].drop_duplicates(), on=["user_id", "data"], how="left", indicator=True)
print("posts de 2026:", len(p26), "| em dia com treino da própria conta:", f"{(m._merge == 'both').mean():.1%}", "| posts por 100 treinos:", round(len(p26) / len(ativ) * 100, 1))

def janela(ref, dias=90):
    ini = ref - pd.Timedelta(days=dias)
    a = ativ[(ativ.data <= ref) & (ativ.data > ini)].groupby("user_id").agg(treinos_90d=("data", "size"), acomp_90d=("acompanhado", "sum"))
    p = posts[(posts.data <= ref) & (posts.data > ini)].groupby("user_id").size().rename("posts_90d")
    return a.join(p, how="outer")

def faixas(d):
    d = d.fillna({"treinos_90d": 0, "acomp_90d": 0, "posts_90d": 0, "n_mutuas": 0})
    d["posts"] = pd.cut(d.posts_90d, [-1, 0, 1, 3, 1000], labels=["0 posts", "1 post", "2-3 posts", "4+ posts"])
    d["volume"] = pd.cut(d.treinos_90d, [-1, 0, 12, 30, 1000], labels=["0 treinos", "1-12", "13-30", "31+"])
    d["rede"] = pd.cut(d.n_mutuas, [-1, 0, 1, 100], labels=["sem mútua", "1 mútua", "2+ mútuas"])
    d["acompanhado"] = np.where(d.acomp_90d > 0, "treinou acompanhado", "não")
    return d

119187 posts de 2025-07-01 a 2026-09-30 | 1252488 treinos | 14112 acompanhados


posts de 2026: 74142 | em dia com treino da própria conta: 77.5% | posts por 100 treinos: 5.9


## 1. Ativos no corte: quantos usam cada parte da rede social

In [2]:
at = assin[assin.ativa_no_corte].merge(usuarios, on="user_id", how="left", validate="one_to_one").merge(eng[["user_id", "nivel", "treinos_30d"]], on="user_id", how="left", validate="one_to_one").merge(rede, on="user_id", how="left", validate="one_to_one")
at = faixas(at.join(janela(CORTE), on="user_id"))
at["casa"] = pd.cut((CORTE - at.inicio).dt.days, [-1, 120, 365, 10000], labels=["até 4 meses", "4-12 meses", "12+ meses"])
print("assinantes ativos:", len(at))
for col in ["posts", "rede", "acompanhado"]:
    t = at[col].value_counts().sort_index().rename("ativos").to_frame(); t["%"] = t.ativos / t.ativos.sum() * 100
    t["% anual"] = at.groupby(col, observed=True).plano_atual.apply(lambda s: (s == "anual").mean() * 100)
    print(f"== {col}"); print(t); print()
at["algum"] = (at.posts_90d > 0) | (at.n_mutuas > 0) | (at.acomp_90d > 0)
print("usa algum dos três:", f"{at.algum.mean():.1%}", "| posta e tem mútua:", f"{((at.posts_90d > 0) & (at.n_mutuas > 0)).mean():.1%}", "| nenhum:", f"{(~at.algum).mean():.1%}")
print("seguidores:", at.n_seguidores.describe()[["mean", "50%", "75%", "max"]].round(1).to_dict(), "| seguindo:", at.n_seguindo.describe()[["mean", "50%", "75%", "max"]].round(1).to_dict())
print("posts em 90 dias (quem postou):", at.loc[at.posts_90d > 0, "posts_90d"].describe()[["mean", "50%", "75%", "max"]].round(1).to_dict())
print("posts por 100 treinos entre os ativos:", round(at.posts_90d.sum() / at.treinos_90d.sum() * 100, 1))
print(); print("rede por tempo de casa (% da linha)"); print(pd.crosstab(at.casa, at.rede, normalize="index") * 100)

assinantes ativos: 14753
== posts
           ativos    %  % anual
posts                          
0 posts      3960 26.8     60.9
1 post       4532 30.7     59.6
2-3 posts    4958 33.6     54.3
4+ posts     1303  8.8     51.3

== rede
           ativos    %  % anual
rede                           
sem mútua    5043 34.2     53.5
1 mútua      4583 31.1     58.5
2+ mútuas    5127 34.8     60.4

== acompanhado
                     ativos    %  % anual
acompanhado                              
não                   13673 92.7     58.3
treinou acompanhado    1080  7.3     46.6

usa algum dos três: 89.8% | posta e tem mútua: 49.7% | nenhum: 10.2%
seguidores: {'mean': 1.7, '50%': 1.0, '75%': 2.0, 'max': 13.0} | seguindo: {'mean': 1.7, '50%': 1.0, '75%': 2.0, 'max': 11.0}
posts em 90 dias (quem postou): {'mean': 2.1, '50%': 2.0, '75%': 3.0, 'max': 9.0}
posts por 100 treinos entre os ativos: 6.0

rede por tempo de casa (% da linha)
rede         sem mútua  1 mútua  2+ mútuas
casa                

## 2. Engajamento dos ativos por uso social

In [3]:
def niveis(d, por):
    n = pd.crosstab(d[por], d.nivel).reindex(columns=NIVEIS, fill_value=0)
    t = (n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos"))
    g = d.groupby(por, observed=True)
    t["treinos 30d"] = g.treinos_30d.mean(); t["treinos 90d"] = g.treinos_90d.mean()
    t["% renovação desligada"] = g.renovacao_automatica.apply(lambda s: (s == "nao").mean() * 100)
    return t
tabs = {}
for col in ["posts", "rede", "acompanhado"]:
    tabs[col] = niveis(at, col); print(f"== por {col}"); print(tabs[col]); print()
print("== MESMO VOLUME DE TREINO: % no nível alto por volume (linhas) e posts (colunas)")
print(at.assign(alto=at.nivel == NIVEIS[0]).pivot_table(index="volume", columns="posts", values="alto", aggfunc="mean", observed=True) * 100)
print(at.pivot_table(index="volume", columns="posts", values="user_id", aggfunc="size", observed=True)); print()
print("== % com renovação desligada por volume e posts")
print(at.assign(d=at.renovacao_automatica == "nao").pivot_table(index="volume", columns="posts", values="d", aggfunc="mean", observed=True) * 100); print()
print("== rede no mesmo tempo de casa: % no nível alto")
print(at.assign(alto=at.nivel == NIVEIS[0]).pivot_table(index="casa", columns="rede", values="alto", aggfunc="mean", observed=True) * 100)
print("treinos em 90 dias (média)"); print(at.pivot_table(index="casa", columns="rede", values="treinos_90d", aggfunc="mean", observed=True))
for pl in ["mensal", "anual"]:
    print(); print(f"== {pl}: por posts"); print(niveis(at[at.plano_atual == pl], "posts"))

== por posts
           1. alto  2. médio  3. baixo  4. possível churn  ativos  treinos 30d  treinos 90d  % renovação desligada
posts                                                                                                             
0 posts       61.0      18.5       3.4               17.1    3960          6.4         14.6                   21.6
1 post        69.2      17.2       2.9               10.7    4532          8.3         21.4                   18.0
2-3 posts     80.2      14.8       1.8                3.2    4958         11.4         31.7                   10.2
4+ posts      89.1       9.6       0.8                0.5    1303         15.9         46.2                    5.4

== por rede
           1. alto  2. médio  3. baixo  4. possível churn  ativos  treinos 30d  treinos 90d  % renovação desligada
rede                                                                                                              
sem mútua     76.1      14.9       2.1                

## 3. Churn por uso social (abr a set/2026)

Sinais medidos nos 90 dias antes do dia 1 de cada mês; rede = mútuas formadas antes do dia 1.

In [4]:
partes = []
for mes in pd.date_range("2026-04-01", "2026-09-01", freq="MS"):
    k = painel[(painel.mes == mes) & painel.ativa_inicio_mes].join(janela(mes - pd.Timedelta(days=1)), on="user_id")
    partes.append(k)
pm = pd.concat(partes, ignore_index=True)
n0 = len(pm)
pm = pm.merge(rede_mes, on=["user_id", "mes"], how="left", validate="one_to_one").merge(usuarios, on="user_id", how="left", validate="many_to_one").merge(assin[["user_id", "inicio"]], on="user_id", how="left", validate="many_to_one")
print("assinatura-mês de abr a set/2026:", n0, "-> depois das junções:", len(pm), "| perdas:", int(pm.perdida_no_mes.sum()))
pm = faixas(pm)
pm["casa"] = pd.cut((pm.mes - pm.inicio).dt.days, [-1, 120, 365, 10000], labels=["até 4 meses", "4-12 meses", "12+ meses"])

def churn(d, por):
    g = d.groupby(por, observed=True).agg(ativas=("user_id", "size"), perdas=("perdida_no_mes", "sum"))
    g["churn %"] = g.perdas / g.ativas * 100
    return g
res = {}
for col in ["posts", "rede", "acompanhado"]:
    c = churn(pm, ["plano", col]); res[col] = c
    print(f"== por {col} e plano"); print(c["churn %"].unstack(0).join(c.ativas.unstack(0), rsuffix=" (ativas)")); print()
pd.concat(res, names=["sinal", "plano", "faixa"]).to_csv(S / "rede_social_churn_por_sinal.csv", encoding="utf-8-sig")
for pl in ["mensal", "anual"]:
    x = churn(pm[pm.plano == pl], ["volume", "posts"])
    print(f"== {pl.upper()}, MESMO VOLUME DE TREINO: churn % por volume (linhas) e posts (colunas)"); print(x["churn %"].unstack()); print(x.ativas.unstack()); print()

assinatura-mês de abr a set/2026: 84868 -> depois das junções: 84868 | perdas: 5542
== por posts e plano
plano      anual  mensal  anual (ativas)  mensal (ativas)
posts                                                    
0 posts      4.6     8.7           13838             8736
1 post       3.7    10.5           15430            11003
2-3 posts    1.9    12.2           15585            12989
4+ posts     0.8    14.8            3905             3382

== por rede e plano
plano      anual  mensal  anual (ativas)  mensal (ativas)
rede                                                     
sem mútua    2.6    11.9           15177            13130
1 mútua      3.3    10.9           15624            11159
2+ mútuas    3.5    10.3           17957            11821

== por acompanhado e plano
plano                anual  mensal  anual (ativas)  mensal (ativas)
acompanhado                                                        
não                    3.3    11.3           45544            32681
trei

In [5]:
print("== conferência só iOS (sem a falha de GPS): MENSAL, churn % por volume e posts")
x = churn(pm[(pm.plano == "mensal") & (pm.sistema == "ios")], ["volume", "posts"]); print(x["churn %"].unstack()); print()
print("== rede no mesmo tempo de casa: churn % por plano")
for pl in ["mensal", "anual"]:
    x = churn(pm[pm.plano == pl], ["casa", "rede"]); print(pl); print(x["churn %"].unstack()); print(x.ativas.unstack()); print()
print("== MENSAL: rede × volume de treino (churn %)")
x = churn(pm[pm.plano == "mensal"], ["volume", "rede"]); print(x["churn %"].unstack()); print()
print("== treino acompanhado × volume (churn %), mensal e anual")
for pl in ["mensal", "anual"]:
    x = churn(pm[pm.plano == pl], ["volume", "acompanhado"]); print(pl); print(x["churn %"].unstack().join(x.ativas.unstack(), rsuffix=" (ativas)")); print()
# quem parou de postar: postava nos 90 dias anteriores à janela e não postou na janela

== conferência só iOS (sem a falha de GPS): MENSAL, churn % por volume e posts
posts      0 posts  1 post  2-3 posts  4+ posts
volume                                         
0 treinos     10.5   100.0      100.0       NaN
1-12          11.1    15.1       18.8      41.7
13-30          6.7     9.0       12.0      17.1
31+            6.3     6.3        8.9      11.8

== rede no mesmo tempo de casa: churn % por plano
mensal
rede         sem mútua  1 mútua  2+ mútuas
casa                                      
até 4 meses       12.6     12.7       14.3
4-12 meses        12.8     12.2       11.6
12+ meses          8.5      7.3        7.5
rede         sem mútua  1 mútua  2+ mútuas
casa                                      
até 4 meses       7711     3539       1753
4-12 meses        3095     4277       5239
12+ meses         2324     3343       4829

anual
rede         sem mútua  1 mútua  2+ mútuas
casa                                      
até 4 meses        0.2      0.0        0.0
4-12 mese

anual
acompanhado  não  treinou acompanhado  não (ativas)  treinou acompanhado (ativas)
volume                                                                           
0 treinos    8.0                  NaN       4,278.0                           NaN
1-12         7.7                  3.6      12,597.0                         468.0
13-30        0.9                  0.6      15,584.0                       1,401.0
31+          0.5                  0.3      13,085.0                       1,345.0



In [6]:
# Postar antecipa a saída? posts nos 30 dias antes da saída × 30 dias anteriores, para quem saiu e para quem ficou (mensal, jun-set)
s = assin.merge(painel[["user_id", "plano"]].drop_duplicates(), on="user_id")
s = s[(s.plano == "mensal")]
sai = s[s.data_fim.between("2026-06-01", "2026-09-30")] if "data_fim" in s else None
a2 = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "plano_atual", "data_fim", "inicio", "ativa_no_corte"])
sai = a2[(a2.plano_atual == "mensal") & a2.data_fim.between("2026-06-01", "2026-09-30") & (a2.inicio <= a2.data_fim - pd.Timedelta(days=90))].assign(ref=lambda d: d.data_fim)
fic = a2[(a2.plano_atual == "mensal") & a2.ativa_no_corte & (a2.inicio <= CORTE - pd.Timedelta(days=90))].assign(ref=CORTE)
def antes(d, nome):
    p = posts.merge(d[["user_id", "ref"]], on="user_id"); t = ativ.merge(d[["user_id", "ref"]], on="user_id")
    r = {}
    for rot, (a, b) in {"61-90 dias antes": (90, 60), "31-60 dias antes": (60, 30), "0-30 dias antes": (30, 0)}.items():
        pp = p[(p.data <= p.ref - pd.Timedelta(days=b)) & (p.data > p.ref - pd.Timedelta(days=a))]
        tt = t[(t.data <= t.ref - pd.Timedelta(days=b)) & (t.data > t.ref - pd.Timedelta(days=a))]
        r[rot] = {"% que postou": pp.user_id.nunique() / len(d) * 100, "posts por conta": len(pp) / len(d), "treinos por conta": len(tt) / len(d), "posts por 100 treinos": len(pp) / max(len(tt), 1) * 100}
    print(f"{nome}: {len(d)} contas"); print(pd.DataFrame(r).T); print()
antes(sai, "MENSAL que saiu de jun a set/2026 (90+ dias de casa)")
antes(fic, "MENSAL ativo no corte (90+ dias de casa)")

MENSAL que saiu de jun a set/2026 (90+ dias de casa): 1730 contas
                  % que postou  posts por conta  treinos por conta  posts por 100 treinos
61-90 dias antes          50.8              0.7               12.0                    5.8
31-60 dias antes          54.9              0.8               10.5                    7.3
0-30 dias antes           42.4              0.5                4.4                   12.6



MENSAL ativo no corte (90+ dias de casa): 4428 contas
                  % que postou  posts por conta  treinos por conta  posts por 100 treinos
61-90 dias antes          43.1              0.6               10.5                    5.4
31-60 dias antes          46.3              0.6               11.4                    5.5
0-30 dias antes           47.3              0.6               11.6                    5.6



## 4. O que os posts dizem: despedida e objetivo cumprido

No mensal, quem posta mais sai mais, mesmo com o mesmo volume de treino. O texto dos posts explica:
parte deles anuncia a saída ou comemora um objetivo cumprido. Classificação por palavras-chave
(aproximada; a extração com IA generativa pode refinar).

In [7]:
pt = pd.read_parquet(P / "feed_posts_limpa.parquet", columns=["user_id", "data", "texto"])
tx = pt.texto.fillna("").str.lower()
DESP = r"cancelei|não renovei|nao renovei|último mês aqui|ultimo mes aqui|dar um tempo do app|pausa nos treinos e no app|depois eu volto"
OBJ = r"conclu[ií]d|completei|terminei|feita ✅|feito ✅|missão cumprida|missao cumprida|valeu cada treino|agora é descansar|agora e descansar|medalha no peito"
pt["tema"] = np.where(tx.str.contains(DESP), "despedida", np.where(tx.str.contains(OBJ), "objetivo cumprido", "treino do dia"))
print("posts por tema (todo o arquivo)"); x = pt.tema.value_counts().rename("posts").to_frame(); x["%"] = x.posts / x.posts.sum() * 100; x["contas"] = pt.groupby("tema").user_id.nunique(); print(x); print()
for t in ["despedida", "objetivo cumprido"]:
    print(f"-- {t}: textos mais comuns"); print(pt[pt.tema == t].texto.value_counts().head(6).to_string()); print()

a3 = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "plano_atual", "data_fim", "inicio", "ativa_no_corte", "motivo_fim"])
def grupo(plano):
    sai = a3[(a3.plano_atual == plano) & a3.data_fim.between("2026-01-01", "2026-09-30")].assign(ref=lambda d: d.data_fim, g="saiu em 2026")
    fic = a3[(a3.plano_atual == plano) & a3.ativa_no_corte].assign(ref=CORTE, g="ativo no corte")
    d = pd.concat([sai, fic])
    k = pt.merge(d[["user_id", "ref", "g"]], on="user_id"); k = k[(k.data <= k.ref) & (k.data > k.ref - pd.Timedelta(days=60))]
    r = d.groupby("g").size().rename("contas").to_frame()
    for t in ["despedida", "objetivo cumprido", "treino do dia"]:
        r[f"% com post de {t}"] = k[k.tema == t].groupby("g").user_id.nunique() / r.contas * 100
    r["% com algum post"] = k.groupby("g").user_id.nunique() / r.contas * 100
    return r.fillna(0)
for plano in ["mensal", "anual"]:
    print(f"== {plano}: posts nos 60 dias antes da saída (ou do corte)"); print(grupo(plano)); print()

# Depois de um post de objetivo cumprido ou de despedida: o que acontece com a assinatura (mensal ativo na data do post)
ev = pt[pt.tema != "treino do dia"].sort_values("data").drop_duplicates(["user_id", "tema"]).merge(a3, on="user_id")
ev = ev[(ev.inicio <= ev.data) & (ev.data_fim.isna() | (ev.data_fim >= ev.data)) & (ev.data <= CORTE - pd.Timedelta(days=60))]
ev["saiu_60d"] = ev.data_fim.notna() & (ev.data_fim <= ev.data + pd.Timedelta(days=60))
g = ev.groupby(["plano_atual", "tema"]).agg(assinantes=("user_id", "size"), saiu_em_60d=("saiu_60d", "sum")); g["% saiu em 60 dias"] = g.saiu_em_60d / g.assinantes * 100
print("primeiro post de cada tema feito por assinante ativo (com 60 dias de observação até o corte)"); print(g); print()
ev["dias_ate_fim"] = (ev.data_fim - ev.data).dt.days
print("dias do post até o fim da assinatura (quem saiu em 60 dias):"); print(ev[ev.saiu_60d].groupby(["plano_atual", "tema"]).dias_ate_fim.describe()[["count", "25%", "50%", "75%"]])
print()
# refazer mensal: churn por posts, sem contar posts de despedida e de objetivo cumprido
pn = pt[pt.tema == "treino do dia"]
partes = []
for mes in pd.date_range("2026-04-01", "2026-09-01", freq="MS"):
    ref = mes - pd.Timedelta(days=1)
    q = pn[(pn.data <= ref) & (pn.data > ref - pd.Timedelta(days=90))].groupby("user_id").size().rename("posts_treino_90d")
    partes.append(pm[pm.mes == mes].join(q, on="user_id"))
pz = pd.concat(partes).fillna({"posts_treino_90d": 0})
pz["posts_treino"] = pd.cut(pz.posts_treino_90d, [-1, 0, 1, 3, 1000], labels=["0 posts", "1 post", "2-3 posts", "4+ posts"])
x = churn(pz[pz.plano == "mensal"], ["volume", "posts_treino"])
print("== MENSAL, só posts de treino do dia: churn % por volume (linhas) e posts (colunas)"); print(x["churn %"].unstack()); print(x.ativas.unstack())

posts por tema (todo o arquivo)
                    posts    %  contas
tema                                  
treino do dia      108003 90.6   26264
objetivo cumprido    9673  8.1    8238
despedida            1511  1.3    1507

-- despedida: textos mais comuns
texto
cancelei o premium, vou correr só com o relógio    223
dando uma pausa nos treinos e no app               208
último mês aqui, valeu galera                      187
acho que vou dar um tempo do app                   184
não renovei, depois eu volto                       164
nao renovei, depois eu volto                        26

-- objetivo cumprido: textos mais comuns
texto
minha meia maratona feita ✅ que dia                  125
medalha no peito, uma prova de 10k concluída         120
completei minha meia maratona!! chorei na chegada    118
completei uma prova de 10k!! chorei na chegada       117
uma prova de 10k feita ✅ que dia                     112
medalha no peito, minha meia maratona concluída      111

== mensal: p

                                count  25%  50%  75%
plano_atual tema                                    
anual       despedida            92.0 33.8 38.0 41.2
            objetivo cumprido   272.0 20.8 37.0 49.0
mensal      despedida           734.0  6.0 12.0 19.0
            objetivo cumprido 1,818.0 24.0 34.0 44.0

== MENSAL, só posts de treino do dia: churn % por volume (linhas) e posts (colunas)
posts_treino  0 posts  1 post  2-3 posts  4+ posts
volume                                            
0 treinos        11.1   100.0      100.0       NaN
1-12             13.1    16.4       20.9      25.0
13-30             8.6    11.2       12.3      13.4
31+               8.8     8.8       10.1      10.9
posts_treino  0 posts  1 post  2-3 posts  4+ posts
volume                                            
0 treinos        36.0     2.0        1.0       NaN
1-12          3,063.0 1,700.0      597.0      16.0
13-30         4,555.0 6,129.0    5,395.0     634.0
31+           1,620.0 3,608.0    6,5